# group tweets into time windows

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("../generated_data/subjects_AND_sampling_metadata.csv", parse_dates=["created_at"])

C:\Users\yuyue\AppData\Local\Temp\ipykernel_30120\3142735918.py:4: DtypeWarning: Columns (3,14,15,16,17,18,26,27,28,29,39,41,42,46,48,49,70,71,72,73,75,76,77,78,79) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("../generated_data/subjects_AND_sampling_metadata.csv", parse_dates=["created_at"])


In [2]:
election_date = pd.Timestamp("2024-11-05", tz="UTC")
start_period = election_date - pd.DateOffset(years=1)
end_period = election_date
df = df[(df["created_at"] >= start_period) & (df["created_at"] <= end_period)].copy()

In [3]:
ELECTION_DATE = pd.Timestamp("2024-11-05").tz_localize("UTC")

df['created_at'] = pd.to_datetime(df['created_at'], utc=True)

# Create 14-day windows starting from first available date
first_date = df['created_at'].min()
df['days_since_start'] = (df['created_at'] - first_date).dt.days
df['window_number'] = (df['days_since_start'] // 14).astype(int)
df['time_window_start'] = first_date + pd.to_timedelta(df['window_number'] * 14, unit='D')
df['window_weeks_to_election'] = ((ELECTION_DATE - df['time_window_start']).dt.days // 7).astype(int)

# Build per-tweet dictionary
df['tweet_dict'] = df.apply(
    lambda row: {
        'text': row['tweets_historical'],
        'created_at': row['created_at'].tz_convert(None)
    },
    axis=1
)

# Group tweets by user and window
grouped = df.groupby(['id', 'time_window_start'])

def combine_tweets(group):
    weeks_to_election = int(group['window_weeks_to_election'].iloc[0])
    tweets_list = group['tweet_dict'].tolist()
    tweets_list.insert(0, f"{weeks_to_election} weeks to election")
    return tweets_list

# Aggregate tweets per window
tweets_df = grouped.apply(combine_tweets).reset_index(name='tweets_struct')

# Get constant user metadata
user_meta_cols = ['id', 'name', 'username', 'description', 'location',
                  'public_metrics.followers_count', 'profile_image_url', 'clean...pool']
user_meta_df = df[user_meta_cols].drop_duplicates(subset=['id'])

# Fill missing windows for all users
all_users = df['id'].unique()
all_windows = pd.date_range(first_date,
                            df['time_window_start'].max(),
                            freq='14D')
full_index = pd.MultiIndex.from_product([all_users, all_windows],
                                        names=['id', 'time_window_start'])
full_df = pd.DataFrame(index=full_index).reset_index()

# Merge tweets per window
full_df = full_df.merge(tweets_df, on=['id', 'time_window_start'], how='left')
full_df = full_df.merge(user_meta_df, on='id', how='left')

# Fill missing tweets_struct with empty list
full_df['tweets_struct'] = full_df['tweets_struct'].apply(lambda x: x if isinstance(x, list) else [])
full_df['is_empty_window'] = full_df['tweets_struct'].apply(lambda x: len(x) == 0)

# Compute weeks to election for all windows
full_df['window_weeks_to_election'] = ((ELECTION_DATE - full_df['time_window_start']).dt.days // 7).astype(int)


C:\Users\yuyue\AppData\Local\Temp\ipykernel_30120\397733186.py:31: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  tweets_df = grouped.apply(combine_tweets).reset_index(name='tweets_struct')


In [4]:
full_df

,id,time_window_start,tweets_struct,name,username,description,location,public_metrics.followers_count,profile_image_url,clean...pool,is_empty_window,window_weeks_to_election
0,1081532502966239232,2023-11-05 00:00:00+00:00,[],caleb ❍,xxaleb,"i think a lot, listen to music. i’m fond of fl...",they/he ☆ 20,4426,https://pbs.twimg.com/profile_images/180336437...,trad,True,52
1,1081532502966239232,2023-11-19 00:00:00+00:00,[],caleb ❍,xxaleb,"i think a lot, listen to music. i’m fond of fl...",they/he ☆ 20,4426,https://pbs.twimg.com/profile_images/180336437...,trad,True,50
2,1081532502966239232,2023-12-03 00:00:00+00:00,[],caleb ❍,xxaleb,"i think a lot, listen to music. i’m fond of fl...",they/he ☆ 20,4426,https://pbs.twimg.com/profile_images/180336437...,trad,True,48
3,1081532502966239232,2023-12-17 00:00:00+00:00,[],caleb ❍,xxaleb,"i think a lot, listen to music. i’m fond of fl...",they/he ☆ 20,4426,https://pbs.twimg.com/profile_images/180336437...,trad,True,46
4,1081532502966239232,2023-12-31 00:00:00+00:00,[],caleb ❍,xxaleb,"i think a lot, listen to music. i’m fond of fl...",they/he ☆ 20,4426,https://pbs.twimg.com/profile_images/180336437...,trad,True,44
...,...,...,...,...,...,...,...,...,...,...,...,...
24123,142827500,2024-08-25 00:00:00+00:00,"[10 weeks to election, {'text': '@foldyrhands ...",Andy,ndrwwwww,architectural designer \n\nhe/him,"Chicago, IL",734,https://pbs.twimg.com/profile_images/137092184...,guns,False,10
24124,142827500,2024-09-08 00:00:00+00:00,"[8 weeks to election, {'text': '@diveomusic :'...",Andy,ndrwwwww,architectural designer \n\nhe/him,"Chicago, IL",734,https://pbs.twimg.com/profile_images/137092184...,guns,False,8
24125,142827500,2024-09-22 00:00:00+00:00,"[6 weeks to election, {'text': '@kevin__fm Sis...",Andy,ndrwwwww,architectural designer \n\nhe/him,"Chicago, IL",734,https://pbs.twimg.com/profile_images/137092184...,guns,False,6
24126,142827500,2024-10-06 00:00:00+00:00,"[4 weeks to election, {'text': '@cherryrepress...",Andy,ndrwwwww,architectural designer \n\nhe/him,"Chicago, IL",734,https://pbs.twimg.com/profile_images/137092184...,guns,False,4


In [5]:
df_with_desc = pd.read_csv("../generated_data/final_user_profiles_cleaned.csv")  

# Merge on "id"
merged_df = full_df.merge(
    df_with_desc[["id", "profile_image_url", "profile_image_description", "profile_image_main_info"]],
    on="id",
    how="left",
    suffixes=("", "_new")  
)

# Overwrite profile_image_url with the new value 
merged_df["profile_image_url"] = merged_df["profile_image_url_new"].combine_first(merged_df["profile_image_url"])

# Drop the temporary column used for overwrite
merged_df = merged_df.drop(columns=["profile_image_url_new"])

merged_df.to_csv("../generated_data/all_user_windows.csv", index=False)
merged_df

,id,time_window_start,tweets_struct,name,username,description,location,public_metrics.followers_count,profile_image_url,clean...pool,is_empty_window,window_weeks_to_election,profile_image_description,profile_image_main_info
0,1081532502966239232,2023-11-05 00:00:00+00:00,[],caleb ❍,xxaleb,"i think a lot, listen to music. i’m fond of fl...",they/he ☆ 20,4426,https://pbs.twimg.com/profile_images/180336437...,trad,True,52,### 1. Visible Physical Characteristics:\n\n* ...,### 1. Visible Physical Characteristics:\n\n* ...
1,1081532502966239232,2023-11-19 00:00:00+00:00,[],caleb ❍,xxaleb,"i think a lot, listen to music. i’m fond of fl...",they/he ☆ 20,4426,https://pbs.twimg.com/profile_images/180336437...,trad,True,50,### 1. Visible Physical Characteristics:\n\n* ...,### 1. Visible Physical Characteristics:\n\n* ...
2,1081532502966239232,2023-12-03 00:00:00+00:00,[],caleb ❍,xxaleb,"i think a lot, listen to music. i’m fond of fl...",they/he ☆ 20,4426,https://pbs.twimg.com/profile_images/180336437...,trad,True,48,### 1. Visible Physical Characteristics:\n\n* ...,### 1. Visible Physical Characteristics:\n\n* ...
3,1081532502966239232,2023-12-17 00:00:00+00:00,[],caleb ❍,xxaleb,"i think a lot, listen to music. i’m fond of fl...",they/he ☆ 20,4426,https://pbs.twimg.com/profile_images/180336437...,trad,True,46,### 1. Visible Physical Characteristics:\n\n* ...,### 1. Visible Physical Characteristics:\n\n* ...
4,1081532502966239232,2023-12-31 00:00:00+00:00,[],caleb ❍,xxaleb,"i think a lot, listen to music. i’m fond of fl...",they/he ☆ 20,4426,https://pbs.twimg.com/profile_images/180336437...,trad,True,44,### 1. Visible Physical Characteristics:\n\n* ...,### 1. Visible Physical Characteristics:\n\n* ...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
24123,142827500,2024-08-25 00:00:00+00:00,"[10 weeks to election, {'text': '@foldyrhands ...",Andy,ndrwwwww,architectural designer \n\nhe/him,"Chicago, IL",734,https://pbs.twimg.com/profile_images/137092184...,guns,False,10,### 1. Visible Physical Characteristics:\n- Th...,### 1. Visible Physical Characteristics:\n- Th...
24124,142827500,2024-09-08 00:00:00+00:00,"[8 weeks to election, {'text': '@diveomusic :'...",Andy,ndrwwwww,architectural designer \n\nhe/him,"Chicago, IL",734,https://pbs.twimg.com/profile_images/137092184...,guns,False,8,### 1. Visible Physical Characteristics:\n- Th...,### 1. Visible Physical Characteristics:\n- Th...
24125,142827500,2024-09-22 00:00:00+00:00,"[6 weeks to election, {'text': '@kevin__fm Sis...",Andy,ndrwwwww,architectural designer \n\nhe/him,"Chicago, IL",734,https://pbs.twimg.com/profile_images/137092184...,guns,False,6,### 1. Visible Physical Characteristics:\n- Th...,### 1. Visible Physical Characteristics:\n- Th...
24126,142827500,2024-10-06 00:00:00+00:00,"[4 weeks to election, {'text': '@cherryrepress...",Andy,ndrwwwww,architectural designer \n\nhe/him,"Chicago, IL",734,https://pbs.twimg.com/profile_images/137092184...,guns,False,4,### 1. Visible Physical Characteristics:\n- Th...,### 1. Visible Physical Characteristics:\n- Th...


In [ ]:
# First, check what type the data actually is
print(f"Type of tweets_struct: {type(full_df['tweets_struct'].iloc[100])}")

# If it's a string, check its length and first few characters
if isinstance(full_df['tweets_struct'].iloc[100], str):
    print(f"String length: {len(full_df['tweets_struct'].iloc[100])}")
    print(f"First 200 characters: {full_df['tweets_struct'].iloc[100]}")

Type of tweets_struct: <class 'list'>


In [7]:
import matplotlib.pyplot as plt

num_users = merged_df["id"].nunique()
print(f"Number of unique users sampled: {num_users}")

num_windows = merged_df.groupby(["id", "time_window_start"]).ngroups
print(f"Number of sampled windows: {num_windows}")


Number of unique users sampled: 928
Number of sampled windows: 24128


In [8]:
# Number of unique windows per user
windows_per_user = full_df.groupby('id')['time_window_start'].nunique()

# Compute stats
avg_windows = windows_per_user.mean()
min_windows = windows_per_user.min()
max_windows = windows_per_user.max()
std_windows = windows_per_user.std()

print(f"Average windows per user: {avg_windows:.2f}")
print(f"Min windows per user: {min_windows}")
print(f"Max windows per user: {max_windows}")
print(f"Standard deviation: {std_windows:.2f}")


Average windows per user: 26.00
Min windows per user: 26
Max windows per user: 26
Standard deviation: 0.00


In [9]:
def count_tweets_by_pattern(x):
    """
    Count tweets by looking for the pattern of tweet dictionaries
    """
    if not isinstance(x, str) or x.strip() in ('[]', ''):
        return 0
    
    # Count occurrences of the tweet pattern: {'text': ... 'created_at': ...}
    tweet_count = x.count("'text':")
    return tweet_count

# Apply pattern counting
full_df['num_tweets'] = full_df['tweets_struct'].apply(count_tweets_by_pattern)

# Results
total_tweets = full_df['num_tweets'].sum()
rows_with_tweets = (full_df['num_tweets'] > 0).sum()
avg_tweets = full_df['num_tweets'].mean()
min_tweets = full_df['num_tweets'].min()
max_tweets = full_df['num_tweets'].max()
std_tweets = full_df['num_tweets'].std()

print(f"Average tweets per window: {avg_tweets:.2f}")
print(f"Min tweets per window: {min_tweets}")
print(f"Max tweets per window: {max_tweets}")
print(f"Standard deviation: {std_tweets:.2f}")

Average tweets per window: 0.00
Min tweets per window: 0
Max tweets per window: 0
Standard deviation: 0.00


In [10]:
full_df

,id,time_window_start,tweets_struct,name,username,description,location,public_metrics.followers_count,profile_image_url,clean...pool,is_empty_window,window_weeks_to_election,num_tweets
0,1081532502966239232,2023-11-05 00:00:00+00:00,[],caleb ❍,xxaleb,"i think a lot, listen to music. i’m fond of fl...",they/he ☆ 20,4426,https://pbs.twimg.com/profile_images/180336437...,trad,True,52,0
1,1081532502966239232,2023-11-19 00:00:00+00:00,[],caleb ❍,xxaleb,"i think a lot, listen to music. i’m fond of fl...",they/he ☆ 20,4426,https://pbs.twimg.com/profile_images/180336437...,trad,True,50,0
2,1081532502966239232,2023-12-03 00:00:00+00:00,[],caleb ❍,xxaleb,"i think a lot, listen to music. i’m fond of fl...",they/he ☆ 20,4426,https://pbs.twimg.com/profile_images/180336437...,trad,True,48,0
3,1081532502966239232,2023-12-17 00:00:00+00:00,[],caleb ❍,xxaleb,"i think a lot, listen to music. i’m fond of fl...",they/he ☆ 20,4426,https://pbs.twimg.com/profile_images/180336437...,trad,True,46,0
4,1081532502966239232,2023-12-31 00:00:00+00:00,[],caleb ❍,xxaleb,"i think a lot, listen to music. i’m fond of fl...",they/he ☆ 20,4426,https://pbs.twimg.com/profile_images/180336437...,trad,True,44,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
24123,142827500,2024-08-25 00:00:00+00:00,"[10 weeks to election, {'text': '@foldyrhands ...",Andy,ndrwwwww,architectural designer \n\nhe/him,"Chicago, IL",734,https://pbs.twimg.com/profile_images/137092184...,guns,False,10,0
24124,142827500,2024-09-08 00:00:00+00:00,"[8 weeks to election, {'text': '@diveomusic :'...",Andy,ndrwwwww,architectural designer \n\nhe/him,"Chicago, IL",734,https://pbs.twimg.com/profile_images/137092184...,guns,False,8,0
24125,142827500,2024-09-22 00:00:00+00:00,"[6 weeks to election, {'text': '@kevin__fm Sis...",Andy,ndrwwwww,architectural designer \n\nhe/him,"Chicago, IL",734,https://pbs.twimg.com/profile_images/137092184...,guns,False,6,0
24126,142827500,2024-10-06 00:00:00+00:00,"[4 weeks to election, {'text': '@cherryrepress...",Andy,ndrwwwww,architectural designer \n\nhe/him,"Chicago, IL",734,https://pbs.twimg.com/profile_images/137092184...,guns,False,4,0


In [11]:
empty_percentage = (full_df['is_empty_window'].mean() * 100).round(2)
print(f"Percentage of empty windows: {empty_percentage}%")


Percentage of empty windows: 56.22%


In [13]:
print(full_df.info())   
print(full_df.head())  
print(full_df.tail()) 


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 24128 entries, 0 to 24127
Data columns (total 13 columns):
 #   Column                          Non-Null Count  Dtype              
---  ------                          --------------  -----              
 0   id                              24128 non-null  int64              
 1   time_window_start               24128 non-null  datetime64[ns, UTC]
 2   tweets_struct                   24128 non-null  object             
 3   name                            24128 non-null  object             
 4   username                        24128 non-null  object             
 5   description                     22984 non-null  object             
 6   location                        24128 non-null  object             
 7   public_metrics.followers_count  24128 non-null  int64              
 8   profile_image_url               24128 non-null  object             
 9   clean...pool                    24128 non-null  object             
 10  is_empty_w

In [15]:
import pandas as pd

In [16]:
merged_df = pd.read_csv("../generated_data/all_user_windows.csv")

In [17]:
merged_df

,id,time_window_start,tweets_struct,name,username,description,location,public_metrics.followers_count,profile_image_url,clean...pool,is_empty_window,window_weeks_to_election,profile_image_description,profile_image_main_info
0,1081532502966239232,2023-11-05 00:00:00+00:00,[],caleb ❍,xxaleb,"i think a lot, listen to music. i’m fond of fl...",they/he ☆ 20,4426,https://pbs.twimg.com/profile_images/180336437...,trad,True,52,### 1. Visible Physical Characteristics:\n\n* ...,### 1. Visible Physical Characteristics:\n\n* ...
1,1081532502966239232,2023-11-19 00:00:00+00:00,[],caleb ❍,xxaleb,"i think a lot, listen to music. i’m fond of fl...",they/he ☆ 20,4426,https://pbs.twimg.com/profile_images/180336437...,trad,True,50,### 1. Visible Physical Characteristics:\n\n* ...,### 1. Visible Physical Characteristics:\n\n* ...
2,1081532502966239232,2023-12-03 00:00:00+00:00,[],caleb ❍,xxaleb,"i think a lot, listen to music. i’m fond of fl...",they/he ☆ 20,4426,https://pbs.twimg.com/profile_images/180336437...,trad,True,48,### 1. Visible Physical Characteristics:\n\n* ...,### 1. Visible Physical Characteristics:\n\n* ...
3,1081532502966239232,2023-12-17 00:00:00+00:00,[],caleb ❍,xxaleb,"i think a lot, listen to music. i’m fond of fl...",they/he ☆ 20,4426,https://pbs.twimg.com/profile_images/180336437...,trad,True,46,### 1. Visible Physical Characteristics:\n\n* ...,### 1. Visible Physical Characteristics:\n\n* ...
4,1081532502966239232,2023-12-31 00:00:00+00:00,[],caleb ❍,xxaleb,"i think a lot, listen to music. i’m fond of fl...",they/he ☆ 20,4426,https://pbs.twimg.com/profile_images/180336437...,trad,True,44,### 1. Visible Physical Characteristics:\n\n* ...,### 1. Visible Physical Characteristics:\n\n* ...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
24123,142827500,2024-08-25 00:00:00+00:00,"['10 weeks to election', {'text': '@foldyrhand...",Andy,ndrwwwww,architectural designer \n\nhe/him,"Chicago, IL",734,https://pbs.twimg.com/profile_images/137092184...,guns,False,10,### 1. Visible Physical Characteristics:\n- Th...,### 1. Visible Physical Characteristics:\n- Th...
24124,142827500,2024-09-08 00:00:00+00:00,"['8 weeks to election', {'text': ""@diveomusic ...",Andy,ndrwwwww,architectural designer \n\nhe/him,"Chicago, IL",734,https://pbs.twimg.com/profile_images/137092184...,guns,False,8,### 1. Visible Physical Characteristics:\n- Th...,### 1. Visible Physical Characteristics:\n- Th...
24125,142827500,2024-09-22 00:00:00+00:00,"['6 weeks to election', {'text': '@kevin__fm S...",Andy,ndrwwwww,architectural designer \n\nhe/him,"Chicago, IL",734,https://pbs.twimg.com/profile_images/137092184...,guns,False,6,### 1. Visible Physical Characteristics:\n- Th...,### 1. Visible Physical Characteristics:\n- Th...
24126,142827500,2024-10-06 00:00:00+00:00,"['4 weeks to election', {'text': '@cherryrepre...",Andy,ndrwwwww,architectural designer \n\nhe/him,"Chicago, IL",734,https://pbs.twimg.com/profile_images/137092184...,guns,False,4,### 1. Visible Physical Characteristics:\n- Th...,### 1. Visible Physical Characteristics:\n- Th...


In [18]:
test_profile = merged_df.iloc[48:50].copy()


In [19]:
test_profile.head()

,id,time_window_start,tweets_struct,name,username,description,location,public_metrics.followers_count,profile_image_url,clean...pool,is_empty_window,window_weeks_to_election,profile_image_description,profile_image_main_info
48,1542629834953719810,2024-09-08 00:00:00+00:00,"['8 weeks to election', {'text': 'What a story...",Ashley Diller,Badash1821,A mom of 5 with common sense! Hates being lie...,"Ohio, USA",625,https://pbs.twimg.com/profile_images/169368304...,radical,False,8,### Step 1: Visible Physical Characteristics\n...,### Step 1: Visible Physical Characteristics\n...
49,1542629834953719810,2024-09-22 00:00:00+00:00,"['6 weeks to election', {'text': ""This is so F...",Ashley Diller,Badash1821,A mom of 5 with common sense! Hates being lie...,"Ohio, USA",625,https://pbs.twimg.com/profile_images/169368304...,radical,False,6,### Step 1: Visible Physical Characteristics\n...,### Step 1: Visible Physical Characteristics\n...


In [20]:
import os
import requests
import json
import time
import re
import random
import unicodedata
from collections import defaultdict

# function to normalize text for parsing
def normalize_text(text):
    # Normalize unicode characters, remove markdown formatting, lowercase, and strip dashes
    if not text:
        return text
    
    text = str(text)
    text = text.replace('*', '').replace('<<', '').replace('>>', '')
    text = re.sub(r'[-\u2010-\u2015\u2212\uFE58\uFE63\uFF0D]', '', text)
    normalized = unicodedata.normalize('NFKD', text)
    ascii_text = normalized.encode('ASCII', 'ignore').decode('ASCII')

    # Clean up whitespace + lowercase
    return ' '.join(ascii_text.split()).lower()

# ------------------------------------------------------------
# 0. load checkpoint
# ------------------------------------------------------------

checkpoint_file = "../generated_data/feature_extraction/checkpoint.json"

def load_checkpoint():
    if os.path.exists(checkpoint_file):
        with open(checkpoint_file, "r", encoding="utf-8") as f:
            return json.load(f)
    return {"last_index": -1} 

def save_checkpoint(last_index):
    with open(checkpoint_file, "w", encoding="utf-8") as f:
        json.dump({"last_index": last_index}, f)

# ------------------------------------------------------------
# 1. Endpoint & authentication
# ------------------------------------------------------------
endpoint_base = "https://aipolling.services.ai.azure.com/models"
api_version = "2024-05-01-preview"
url = f"{endpoint_base}/chat/completions?api-version={api_version}"

# Load API key from environment variable
api_key = os.getenv("AZURE_API_KEY")
if not api_key:
    raise ValueError("AZURE_API_KEY environment variable is not set")


# ------------------------------------------------------------
# 2. Load belief categories from JSON
# ------------------------------------------------------------
with open("../generated_data/belief_features.json", "r", encoding="utf-8") as f:
    feature_data = json.load(f)

demo_features = feature_data["demo"]
belief_features = feature_data["beliefs"]

# Normalize titles for the matching
for demo in demo_features:
    demo["title"] = normalize_text(demo["title"])
for belief in belief_features:
    belief["title"] = normalize_text(belief["title"])

# ------------------------------------------------------------
# 3. Base prompt
# ------------------------------------------------------------
prompt_template = """
A social media account has the following username, name, description:
- Name: {name}  
- Username: {username}  
- Description: {description}  
A structured description of their profile image is the following:
- Profile Image Description: {profile_image_description}
Furthermore, they self-report their location in their bio as: {location}  
The amount of followers they have is: {followers}  
They have written the following tweet(s): {tweets}  

I will show you a number of categories to which this user may belong to.
The categories are preceded by a title (e.g. "AGE:" or "SEX:" etc.) and a symbol (e.g. "A1", "A2" or "E1" etc.).
Please select, for each title, the most likely category to which this user belongs to.

In your answer present, for each title, the selected symbol.
Write out in full the category associated with the selected symbol.
The chosen symbol / category must be the most likely to accurately represent this user.
You must only select one symbol / category per title.
A title, symbol and category cannot appear more than once in your answer.

For each selected symbol / category, please note the level of Speculation involved in this selection.
Present the Speculation level for each selection on a scale from 0 (not speculative at all, every single element of the user data was useful in the selection) to 100 (fully speculative, there is no information related to this title in the user data).
Speculation levels should be a direct measure of the amount of useful information available in the user data.
Speculation levels pertain only to the information available in the user data -- namely the username, name, description, location, profile picture and tweets from this user -- and should not be affected by additional information available to you from any other source.

To ensure consistency, use the following guidelines to determine speculation levels:

0-20 (Low speculation): The user data provides clear and direct information relevant to the title. (e.g., explicit mention in the profile or tweets)
21-40 (Moderate-low speculation): The user data provides indirect but strong indicators relevant to the title.
41-60 (Moderate speculation): The user data provides some hints or partial information relevant to the title.
61-80 (Moderate-high speculation): The user data provides limited and weak indicators relevant to the title.
81-100 (High speculation): The user data provides no or almost no information relevant to the title.

For each selected category, please explain at length what features of the data contributed to your choice and your speculation level.

Preserve a strictly structured answer to ease parsing of the text.
Format your output as follows (this is just an example):

title: AGE
explanation: ...
symbol: A1)
category: 18-25
speculation: 90

For every title, you must include exactly these 5 fields: title, explanation, symbol, category, speculation. Do not omit any of them.
Please keep the EXACT WORDING for the titles!
YOU MUST GIVE AN ANSWER FOR EVERY TITLE !
    """

# ------------------------------------------------------------
# 4. Build dynamic belief prompt
# ------------------------------------------------------------
def build_belief_prompt(user_row, beliefs):
    
    #randomly shuffle the belives to prevent position bias
    shuffled_beliefs = beliefs[:]
    random.shuffle(shuffled_beliefs)

    categories_text = "\n".join(
        [
            f"- **{belief['title']}:** \n    Choose one: [\n    " +
            ", \n    ".join([f"{key}) {val}" for key, val in belief["options"].items()]) +
            "]"
            for belief in shuffled_beliefs
        ]
    )

    return prompt_template.format(
        name=user_row['name'],
        username=user_row['username'],
        description=user_row['description'],
        location=user_row['location'],
        followers=user_row['public_metrics.followers_count'],
        tweets=user_row['tweets_struct'],
        profile_image_description=user_row['profile_image_main_info'],
    ) + "\n\nBelow is the list of categories to which this user may belong to:\n" + categories_text

# ----------------------------
# 5. Result Parsing Function
# ----------------------------

def parse_belief_response(response, beliefs, prefix=None):
    categories = {}
    explanations = {}
    speculations = {}
    symbols = {}

    for belief in beliefs:
        safe_key = re.sub(r"\W+", "_", belief["title"])
        if prefix:
            safe_key = f"{prefix}_{safe_key}"
        categories[f"{safe_key}_Category"] = None
        explanations[f"{safe_key}_Explanation"] = None
        symbols[f"{safe_key}_Symbol"] = None
        speculations[f"{safe_key}_Speculation"] = None

    belief_sections = re.split(r"(?=title:\s*)", str(response), flags=re.IGNORECASE)
    belief_sections = [section.strip() for section in belief_sections if section.strip() and not section.lower().startswith('below')]
    print(f"[DEBUG] Found {len(belief_sections)} sections to parse") 
    
    # Pattern for individual belief
    pattern = (
        r"title:?\s*(.*?)\s*[\r\n]"
        r"explanation:?\s*(.*?)\s*[\r\n]"
        r"symbol:?\s*(.*?)\s*[\r\n]"
        r"category:?\s*(.*?)\s*[\r\n]"
        r"speculation:?\s*(\d+)\s*"
    )
    
    for i, section in enumerate(belief_sections):
        
        match = re.search(pattern, section, re.DOTALL | re.IGNORECASE)
        
        if match:
            title = normalize_text(match.group(1))
            explanation = normalize_text(match.group(2))
            symbol = normalize_text(match.group(3))
            category = normalize_text(match.group(4))
            speculation = normalize_text(match.group(5))
            
            # Find best matching belief title from JSON
            matched_belief = None
            for b in beliefs:
                if b["title"] in title or title in b["title"]:
                    matched_belief = b
                    break
            if matched_belief:
                safe_key = re.sub(r"\W+", "_", matched_belief["title"])
                if prefix:
                    safe_key = f"{prefix}_{safe_key}"
                categories[f"{safe_key}_Category"] = category.strip()
                explanations[f"{safe_key}_Explanation"] = explanation.strip()
                symbols[f"{safe_key}_Symbol"] = symbol.strip()
                speculations[f"{safe_key}_Speculation"] = int(speculation.strip())

    return {**categories, **explanations, **speculations, **symbols}


# ------------------------------------------------------------
# 6. Build the request body
# ------------------------------------------------------------
def infer_beliefs(row, model_name, features):

    prompt = build_belief_prompt(row, features)

    messages = [
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": prompt}
    ]

    body = {
        "model": model_name,
        "max_tokens": 16384,
        "messages": messages
    }

    headers = {
        "Content-Type": "application/json",
        "api-key": api_key
    }

    response = requests.post(url, headers=headers, data=json.dumps(body))

    response.raise_for_status()

    content_json = response.json()
    reply = content_json["choices"][0]["message"]["content"]
    return reply

# ------------------------------------------------------------
# 7. Secondary GPT cleanup function
# ------------------------------------------------------------
cleanup_prompt_template = """
You will be shown a long-form response to a question asking to assign a series of categories to a user.
Your task is to extract and reformat ONLY the belief category assessments into a strict format.

CRITICAL FORMATTING RULES:
- Each belief must be formatted EXACTLY as follows:
title: <FULL TITLE TEXT>
explanation: <FULL EXPLANATION TEXT>
symbol: <SYMBOL CODE>
category: <FULL CATEGORY TEXT>
speculation: <NUMERIC VALUE ONLY>

- Each field must be on its own separate line
- Preserve ALL original content from explanations, categories, and titles - do not summarize or shorten
- For speculation, use ONLY the numeric value (e.g., "50" not "fifty" or "50%")
- Remove any asterisks, markdown, or extra formatting from the original
- Ensure there is exactly one blank line between each complete belief assessment
- If any field is missing, include the field name with nothing after it (e.g., "explanation:")

CHARACTER AND ENCODING RULES:
- Use only standard ASCII hyphens (-) for all dashes and hyphens in titles and categories
- Replace any special dashes (‑, –, —) with regular hyphens (-)
- Ensure all text uses standard ASCII characters only
- Remove any non-ASCII characters or convert them to their closest ASCII equivalent
- Use regular parentheses () not special parentheses

EXAMPLES OF CORRECT FORMATTING:

Example 1:
title: OPINION ON WHETHER DIVORCE SHOULD BE EASIER OR MORE DIFFICULT
explanation: The user's tweet mentions divorce in a positive context...
symbol: SoA1)
category: Obtaining a divorce should be easier than it is
speculation: 50

Example 2:
title: POLITICAL AFFILIATION
explanation: The user frequently praises conservative figures...
symbol: PoA3)
category: Republican
speculation: 75

COMMON MISTAKES TO AVOID:
- WRONG: speculation: fifty (50)
- CORRECT: speculation: 50

- WRONG: symbol: SoA1) (with extra spaces)
- CORRECT: symbol: SoA1)

- WRONG: Using markdown like **title:**
- CORRECT: title:

- WRONG: Combining multiple fields on one line
- CORRECT: Each field on its own line

- WRONG: title: OPINION ON ISRAEL‑HAMAS WAR (special hyphen)
- CORRECT: title: OPINION ON ISRAEL-HAMAS WAR (regular hyphen)

- WRONG: category: Obtaining a divorce should be easier than it is (special characters)
- CORRECT: category: Obtaining a divorce should be easier than it is

Now, clean and reformat the following response to match this exact structure. Preserve all original meaning but enforce the strict formatting and character rules:

{raw_output}
"""

def clean_gpt_output(raw_output, model_name):
    messages = [
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": cleanup_prompt_template.format(raw_output=raw_output)}
    ]

    body = {
        "model": model_name,
        "max_tokens": 16384,
        "messages": messages
    }

    headers = {
        "Content-Type": "application/json",
        "api-key": api_key
    }

    response = requests.post(url, headers=headers, data=json.dumps(body))
    response.raise_for_status()
    content_json = response.json()
    reply = content_json["choices"][0]["message"]["content"]
    return reply

# ------------------------------------------------------------
# 8. GPT call with retry logic
# ------------------------------------------------------------
def infer_beliefs_with_retry(row, model_name, features, prefix=None, max_retries=15):
    for attempt in range(1, max_retries + 1):
        response_text = None
        cleaned_text = None
        try:
            response_text = infer_beliefs(row, model_name, features)
            parsed_result = parse_belief_response(response_text, features, prefix=prefix)

            # if parsed result is incomplete, proceed with seecondary cleanup
            if any(value is None for value in parsed_result.values()):
                print(f"[DEBUG] Parsing incomplete for {row['username']} → attempting cleanup.")
                cleaned_text = clean_gpt_output(response_text, model_name)
                parsed_result = parse_belief_response(cleaned_text, features, prefix=prefix)

            # if still incomplete, raise exception to retry full GPT
            if any(value is None for value in parsed_result.values()):
                raise ValueError("Parsed result is empty. Retrying...")
            else:
                print(f"Parsing succeeded directly for {row['username']}")
            return response_text, cleaned_text, parsed_result

        except Exception as e:
            print(f"Attempt {attempt}/{max_retries} failed for user {row['username']} ({model_name}): {e}")
            if attempt < max_retries:
                time.sleep(2 ** attempt * 0.1)  
    return response_text, cleaned_text, parsed_result

# ------------------------------------------------------------
# 9. Apply inference to the dataframe
# ------------------------------------------------------------
models = [
    "gpt-oss-120b"
]

df = merged_df
all_results = []
user_results = defaultdict(list)

output_file = "../generated_data/feature_extraction/all_results.csv"
user_dir = "../generated_data/feature_extraction/users"
os.makedirs(user_dir, exist_ok=True)

# load results from checkpoint
if os.path.exists(output_file):
    all_results = pd.read_csv(output_file, encoding="utf-8-sig").to_dict("records")

checkpoint = load_checkpoint()
start_index = checkpoint["last_index"] + 1
print(f"Resuming: {start_index} rows already processed.")

completed_users = set()
total_users = df['id'].nunique()

for model_name in models:
    current_user = None
    
    for index, row in df.iloc[start_index:].iterrows():
        user_id = row["id"]

        if current_user != user_id:
            if current_user is not None:
                # Save completed previous user
                pd.DataFrame(user_results[current_user]).to_csv(
                    f"{user_dir}/{current_user}.csv", index=False, encoding="utf-8-sig"
                )
                print(f"Finished all timewindows for user {current_user}")
                completed_users.add(current_user)
                print(f"{len(completed_users)}/{total_users} users completed")

            # Switch to new user
            current_user = user_id

            # Load existing user CSV if exists to resume
            user_file = f"{user_dir}/{user_id}.csv"
            if os.path.exists(user_file):
                user_results[user_id] = pd.read_csv(user_file, encoding="utf-8-sig").to_dict("records")
            else:
                user_results[user_id] = []
        try:
            combined_row = row.to_dict()
            # with empty windows, set all belief/demo outputs to None
            if row.get("is_empty_window", False):
                combined_row = row.to_dict()
                combined_row.update({
                    "Model": model_name,
                    "Belief_Raw": None,
                    "Belief_Cleaned": None,
                    "Demo_Raw": None,
                    "Demo_Cleaned": None
                })

                for feature in belief_features:
                    safe_key = re.sub(r"\W+", "_", feature["title"])
                    combined_row.update({
                        f"Belief_{safe_key}_Category": None,
                        f"Belief_{safe_key}_Explanation": None,
                        f"Belief_{safe_key}_Symbol": None,
                        f"Belief_{safe_key}_Speculation": None
                    })
                for feature in demo_features:
                    safe_key = re.sub(r"\W+", "_", feature["title"])
                    combined_row.update({
                        f"Demo_{safe_key}_Category": None,
                        f"Demo_{safe_key}_Explanation": None,
                        f"Demo_{safe_key}_Symbol": None,
                        f"Demo_{safe_key}_Speculation": None
                    })

                all_results.append(combined_row)
                user_results[user_id].append(combined_row)
                save_checkpoint(index)
                continue 
            # Belief features
            belief_text, cleaned_belief, parsed_belief = infer_beliefs_with_retry(row, model_name, belief_features, prefix="Belief")

            # Demo features
            demo_text, cleaned_demo, parsed_demo = infer_beliefs_with_retry(row, model_name, demo_features, prefix="Demo")

            combined_row.update({
                "Model": model_name,
                "Belief_Raw": belief_text,
                "Belief_Cleaned": cleaned_belief,
                "Demo_Raw": demo_text,
                "Demo_Cleaned": cleaned_demo
            })
            combined_row.update(parsed_belief)
            combined_row.update(parsed_demo)
            all_results.append(combined_row)
            user_results[user_id].append(combined_row)

            pd.DataFrame(all_results).to_csv("../generated_data/feature_extraction/all_results.csv", index=False, encoding="utf-8-sig")

            pd.DataFrame(user_results[user_id]).to_csv(
                f"{user_dir}/{user_id}.csv", index=False, encoding="utf-8-sig"
            )

            if (index + 1) % 50 == 0:
                print(f"Progess: {index + 1}/{len(df)} rows completed")
            save_checkpoint(index)
            time.sleep(1)  
        except Exception as e:
            print(f"Error processing row {index} for model {model_name}: {e}")

# for the last user
if current_user is not None and current_user not in completed_users:
    pd.DataFrame(user_results[current_user]).to_csv(
        f"{user_dir}/{current_user}.csv", index=False, encoding="utf-8-sig"
    )
    print(f"Finished all timewindows for user {current_user}")
    completed_users.add(current_user)
    print(f"{len(completed_users)}/{total_users} users completed")

print("Done!")


Resuming: 24128 rows already processed.
Done!


In [31]:
# Example: print the constructed prompt for one user, to sanity-check formatting
example_row = df[~df.get("is_empty_window", False)].iloc[200]
example_prompt = build_belief_prompt(example_row, belief_features)
print(example_prompt)


A social media account has the following username, name, description:
- Name: Peter Sachon  
- Username: carbonmando  
- Description: Cello, Mandocello, Octave Mandolin.  
A structured description of their profile image is the following:
- Profile Image Description: ### 1. Visible Physical Characteristics:

- **Skin Tone**: Fair or light.
- **Hair Style**: Short, neatly combed gray hair.
- **Facial Hair**: Clean-shaven.
- **Wrinkles**: Visible around the eyes, suggesting age.
- **Clothing**: Red sweater over a light blue collared shirt.

### 2. Visible Contextual or Lifestyle Cues:

- **Background**: Blurred indoor setting, possibly a living room or office.
- **Objects/Background Elements**: Unclear due to blurriness, but there are hints of furniture and possibly a lit lamp or warm lighting.

### 3. Demographic Information:

- **Approximate Age Range**: Likely between 45 and 60 years old due to the gray hair and visible wrinkles.
- **Gender Presentation**: Male.
- **Racial or Ethnic B

In [22]:
import pandas as pd

In [23]:
user_df = pd.read_csv("../generated_data/feature_extraction/users/1542629834953719810.csv")

In [24]:
user_df

,id,time_window_start,tweets_struct,name,username,description,location,public_metrics.followers_count,profile_image_url,clean...pool,...,Demo_interest_in_politics_Symbol,Demo_interest_in_politics_Speculation,Demo_marital_status_Category,Demo_marital_status_Explanation,Demo_marital_status_Symbol,Demo_marital_status_Speculation,Demo_partisan_loyalties_Category,Demo_partisan_loyalties_Explanation,Demo_partisan_loyalties_Symbol,Demo_partisan_loyalties_Speculation
0,1542629834953719810,2023-11-05 00:00:00+00:00,[],Ashley Diller,Badash1821,A mom of 5 with common sense! Hates being lie...,"Ohio, USA",625,https://pbs.twimg.com/profile_images/169368304...,radical,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1542629834953719810,2023-11-19 00:00:00+00:00,[],Ashley Diller,Badash1821,A mom of 5 with common sense! Hates being lie...,"Ohio, USA",625,https://pbs.twimg.com/profile_images/169368304...,radical,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1542629834953719810,2023-12-03 00:00:00+00:00,[],Ashley Diller,Badash1821,A mom of 5 with common sense! Hates being lie...,"Ohio, USA",625,https://pbs.twimg.com/profile_images/169368304...,radical,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1542629834953719810,2023-12-17 00:00:00+00:00,[],Ashley Diller,Badash1821,A mom of 5 with common sense! Hates being lie...,"Ohio, USA",625,https://pbs.twimg.com/profile_images/169368304...,radical,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,1542629834953719810,2023-12-31 00:00:00+00:00,[],Ashley Diller,Badash1821,A mom of 5 with common sense! Hates being lie...,"Ohio, USA",625,https://pbs.twimg.com/profile_images/169368304...,radical,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,1542629834953719810,2024-01-14 00:00:00+00:00,[],Ashley Diller,Badash1821,A mom of 5 with common sense! Hates being lie...,"Ohio, USA",625,https://pbs.twimg.com/profile_images/169368304...,radical,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,1542629834953719810,2024-01-28 00:00:00+00:00,[],Ashley Diller,Badash1821,A mom of 5 with common sense! Hates being lie...,"Ohio, USA",625,https://pbs.twimg.com/profile_images/169368304...,radical,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,1542629834953719810,2024-02-11 00:00:00+00:00,[],Ashley Diller,Badash1821,A mom of 5 with common sense! Hates being lie...,"Ohio, USA",625,https://pbs.twimg.com/profile_images/169368304...,radical,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,1542629834953719810,2024-02-25 00:00:00+00:00,[],Ashley Diller,Badash1821,A mom of 5 with common sense! Hates being lie...,"Ohio, USA",625,https://pbs.twimg.com/profile_images/169368304...,radical,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,1542629834953719810,2024-03-10 00:00:00+00:00,[],Ashley Diller,Badash1821,A mom of 5 with common sense! Hates being lie...,"Ohio, USA",625,https://pbs.twimg.com/profile_images/169368304...,radical,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [25]:
final_df = pd.read_csv("../generated_data/feature_extraction/all_results.csv")

KeyboardInterrupt: 

In [ ]:
final_df

,id,time_window_start,tweets_struct,name,username,description,location,public_metrics.followers_count,profile_image_url,clean...pool,...,Demo_interest_in_politics_Symbol,Demo_interest_in_politics_Speculation,Demo_marital_status_Category,Demo_marital_status_Explanation,Demo_marital_status_Symbol,Demo_marital_status_Speculation,Demo_partisan_loyalties_Category,Demo_partisan_loyalties_Explanation,Demo_partisan_loyalties_Symbol,Demo_partisan_loyalties_Speculation
0,1081532502966239232,2023-11-05 00:00:00+00:00,[],caleb ❍,xxaleb,"i think a lot, listen to music. i’m fond of fl...",they/he ☆ 20,4426,https://pbs.twimg.com/profile_images/180336437...,trad,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1081532502966239232,2023-11-19 00:00:00+00:00,[],caleb ❍,xxaleb,"i think a lot, listen to music. i’m fond of fl...",they/he ☆ 20,4426,https://pbs.twimg.com/profile_images/180336437...,trad,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1081532502966239232,2023-12-03 00:00:00+00:00,[],caleb ❍,xxaleb,"i think a lot, listen to music. i’m fond of fl...",they/he ☆ 20,4426,https://pbs.twimg.com/profile_images/180336437...,trad,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1081532502966239232,2023-12-17 00:00:00+00:00,[],caleb ❍,xxaleb,"i think a lot, listen to music. i’m fond of fl...",they/he ☆ 20,4426,https://pbs.twimg.com/profile_images/180336437...,trad,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,1081532502966239232,2023-12-31 00:00:00+00:00,[],caleb ❍,xxaleb,"i think a lot, listen to music. i’m fond of fl...",they/he ☆ 20,4426,https://pbs.twimg.com/profile_images/180336437...,trad,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,1081532502966239232,2024-01-14 00:00:00+00:00,[],caleb ❍,xxaleb,"i think a lot, listen to music. i’m fond of fl...",they/he ☆ 20,4426,https://pbs.twimg.com/profile_images/180336437...,trad,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,1081532502966239232,2024-01-28 00:00:00+00:00,[],caleb ❍,xxaleb,"i think a lot, listen to music. i’m fond of fl...",they/he ☆ 20,4426,https://pbs.twimg.com/profile_images/180336437...,trad,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,1081532502966239232,2024-02-11 00:00:00+00:00,[],caleb ❍,xxaleb,"i think a lot, listen to music. i’m fond of fl...",they/he ☆ 20,4426,https://pbs.twimg.com/profile_images/180336437...,trad,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,1081532502966239232,2024-02-25 00:00:00+00:00,[],caleb ❍,xxaleb,"i think a lot, listen to music. i’m fond of fl...",they/he ☆ 20,4426,https://pbs.twimg.com/profile_images/180336437...,trad,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,1081532502966239232,2024-03-10 00:00:00+00:00,[],caleb ❍,xxaleb,"i think a lot, listen to music. i’m fond of fl...",they/he ☆ 20,4426,https://pbs.twimg.com/profile_images/180336437...,trad,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
test = final_df.loc[10:15, ["username", "Belief_Raw", "Belief_Cleaned"]]
test

,username,Belief_Raw,Belief_Cleaned
10,xxaleb,**title: OPINION ON WHICH SIDE HAS MORE VALID ...,NaN
11,xxaleb,**title: OPINION ON WHETHER DIVORCE IN THE USA...,NaN
12,xxaleb,**title: OPINION ON WHICH SIDE HAS MORE VALID ...,NaN
13,xxaleb,**title: OPINION ON WHICH SIDE HAS MORE VALID ...,NaN
14,xxaleb,**title: OPINION ON WHETHER DIVORCE IN THE USA...,NaN
15,xxaleb,**title: OPINION ON WHICH SIDE HAS MORE VALID ...,NaN


In [ ]:
test = final_df[["username", "Belief_Raw", "Belief_Cleaned"]]
test

,username,Belief_Raw,Belief_Cleaned
0,xxaleb,NaN,NaN
1,xxaleb,NaN,NaN
2,xxaleb,NaN,NaN
3,xxaleb,NaN,NaN
4,xxaleb,NaN,NaN
5,xxaleb,NaN,NaN
6,xxaleb,NaN,NaN
7,xxaleb,NaN,NaN
8,xxaleb,NaN,NaN
9,xxaleb,NaN,NaN


In [ ]:
final_df

,id,time_window_start,tweets_struct,name,username,description,location,public_metrics.followers_count,profile_image_url,clean...pool,...,Belief_OPINION_ON_WHETHER_DIVORCE_IN_THE_USA_SHOULD_BE_EASIER_OR_MORE_DIFFICULT_TO_OBTAIN_THAN_IT_IS_Symbol,Belief_OPINION_ON_WHICH_SIDE_HAS_MORE_VALID_REASONS_IN_THE_ISRAEL_HAMAS_WAR_Symbol,Demo_AGE_Category,Demo_SEX_Category,Demo_AGE_Explanation,Demo_SEX_Explanation,Demo_AGE_Speculation,Demo_SEX_Speculation,Demo_AGE_Symbol,Demo_SEX_Symbol
0,1081532502966239232,2024-08-11 00:00:00+00:00,"['12 weeks to election', {'text': '@mitskimarr...",caleb ❍,xxaleb,"i think a lot, listen to music. i’m fond of fl...",they/he ☆ 20,4426,https://pbs.twimg.com/profile_images/180336437...,trad,...,SoA3),IsA3),18 to 24 years old,masculine sex – male,The user’s self‑reported location line reads “...,The same self‑report line includes the pronoun...,10.0,15.0,A2),S1)
1,1081532502966239232,2024-08-25 00:00:00+00:00,"['10 weeks to election', {'text': 'its so funn...",caleb ❍,xxaleb,"i think a lot, listen to music. i’m fond of fl...",they/he ☆ 20,4426,https://pbs.twimg.com/profile_images/180336437...,trad,...,SoA1),IsA4),18 to 24 years old,masculine sex – male,The user’s self‑reported location line reads “...,"The user lists pronouns “they/he”, which inclu...",10.0,30.0,A2),S1)
2,1081532502966239232,2024-09-08 00:00:00+00:00,"['8 weeks to election', {'text': 'she has take...",caleb ❍,xxaleb,"i think a lot, listen to music. i’m fond of fl...",they/he ☆ 20,4426,https://pbs.twimg.com/profile_images/180336437...,trad,...,SoA3),IsA4),18 to 24 years old,masculine sex - male,"The user’s bio includes “they/he ☆ 20”, which ...",The profile picture and description suggest a ...,10.0,25.0,A2),S1)
3,1081532502966239232,2024-09-22 00:00:00+00:00,"['6 weeks to election', {'text': 'i think divo...",caleb ❍,xxaleb,"i think a lot, listen to music. i’m fond of fl...",they/he ☆ 20,4426,https://pbs.twimg.com/profile_images/180336437...,trad,...,SoA1),IsA3),18 to 24 years old**,masculine sex - male**,The same self‑reported location line includes ...,The user’s self‑reported location line reads “...,10.0,15.0,A2)**,S1)**
4,1081532502966239232,2024-10-06 00:00:00+00:00,[],caleb ❍,xxaleb,"i think a lot, listen to music. i’m fond of fl...",they/he ☆ 20,4426,https://pbs.twimg.com/profile_images/180336437...,trad,...,None,None,None,None,None,None,NaN,NaN,None,None
5,1081532502966239232,2024-10-20 00:00:00+00:00,[],caleb ❍,xxaleb,"i think a lot, listen to music. i’m fond of fl...",they/he ☆ 20,4426,https://pbs.twimg.com/profile_images/180336437...,trad,...,None,None,None,None,None,None,NaN,NaN,None,None
6,1542629834953719810,2023-11-05 00:00:00+00:00,[],Ashley Diller,Badash1821,A mom of 5 with common sense! Hates being lie...,"Ohio, USA",625,https://pbs.twimg.com/profile_images/169368304...,radical,...,None,None,None,None,None,None,NaN,NaN,None,None
7,1542629834953719810,2023-11-19 00:00:00+00:00,[],Ashley Diller,Badash1821,A mom of 5 with common sense! Hates being lie...,"Ohio, USA",625,https://pbs.twimg.com/profile_images/169368304...,radical,...,None,None,None,None,None,None,NaN,NaN,None,None
8,1542629834953719810,2023-12-03 00:00:00+00:00,[],Ashley Diller,Badash1821,A mom of 5 with common sense! Hates being lie...,"Ohio, USA",625,https://pbs.twimg.com/profile_images/169368304...,radical,...,None,None,None,None,None,None,NaN,NaN,None,None
9,1542629834953719810,2023-12-17 00:00:00+00:00,[],Ashley Diller,Badash1821,A mom of 5 with common sense! Hates being lie...,"Ohio, USA",625,https://pbs.twimg.com/profile_images/169368304...,radical,...,None,None,None,None,None,None,NaN,NaN,None,None
